## Example 1: The compiled fastbinary codec

In [ ]:
import importlib.metadata

from thrift.protocol import fastbinary
from thrift.protocol.TBinaryProtocol import TBinaryProtocolAcceleratedFactory
from thrift.TSerialization import deserialize, serialize

from thrift.Thrift import TType
class Point:  # a hand-written struct in the shape `thrift --gen py` emits
    thrift_spec = (None, (1, TType.I32, "x", None, None), (2, TType.STRING, "name", "UTF8", None))
    def __init__(self, x=None, name=None): self.x, self.name = x, name
    def write(self, p): p.trans.write(p._fast_encode(self, [Point, self.thrift_spec]))
    def read(self, p): p._fast_decode(self, p, [Point, self.thrift_spec])

# Example 1: a struct through the C++ accelerated binary protocol (thrift.protocol.fastbinary)
assert importlib.metadata.version("thrift") == "0.25.0", importlib.metadata.version("thrift")
assert hasattr(fastbinary, "encode_binary")
factory = TBinaryProtocolAcceleratedFactory()
blob = serialize(Point(7, "seven"), protocol_factory=factory)
back = deserialize(Point(), blob, protocol_factory=factory)
assert (back.x, back.name) == (7, "seven"), vars(back)
print("Example 1 OK:", len(blob), "bytes")

## Example 2: The compact protocol, accelerated

In [ ]:
from thrift.protocol.TCompactProtocol import TCompactProtocolAcceleratedFactory
from thrift.TSerialization import deserialize, serialize

from thrift.Thrift import TType
class Point:  # a hand-written struct in the shape `thrift --gen py` emits
    thrift_spec = (None, (1, TType.I32, "x", None, None), (2, TType.STRING, "name", "UTF8", None))
    def __init__(self, x=None, name=None): self.x, self.name = x, name
    def write(self, p): p.trans.write(p._fast_encode(self, [Point, self.thrift_spec]))
    def read(self, p): p._fast_decode(self, p, [Point, self.thrift_spec])

# Example 2: the same struct, compact encoding (varints, smaller on the wire)
factory = TCompactProtocolAcceleratedFactory()
blob = serialize(Point(300, "x" * 20), protocol_factory=factory)
back = deserialize(Point(), blob, protocol_factory=factory)
assert (back.x, back.name) == (300, "x" * 20)
print("Example 2 OK:", len(blob), "bytes")

## Example 3: Writing and reading fields by hand on a memory transport

In [ ]:
from thrift.protocol.TBinaryProtocol import TBinaryProtocol
from thrift.Thrift import TType
from thrift.transport.TTransport import TMemoryBuffer

# Example 3: the pure-Python protocol API, the layer generated code calls
out = TMemoryBuffer()
p = TBinaryProtocol(out)
p.writeListBegin(TType.DOUBLE, 3)
for v in (1.5, 2.5, 3.5):
    p.writeDouble(v)
p.writeListEnd()
p.writeString("tail")
inp = TBinaryProtocol(TMemoryBuffer(out.getvalue()))
etype, size = inp.readListBegin()
values = [inp.readDouble() for _ in range(size)]
assert (etype, values, inp.readString()) == (TType.DOUBLE, [1.5, 2.5, 3.5], "tail")
print("Example 3 OK:", values)

## Example 4: JSON protocol

In [ ]:
import json

from thrift.protocol.TJSONProtocol import TJSONProtocol
from thrift.transport.TTransport import TMemoryBuffer

# Example 4: Thrift's JSON wire format, readable as JSON
out = TMemoryBuffer()
p = TJSONProtocol(out)
p.writeMapBegin(11, 8, 2)  # TType.STRING -> TType.I32
for k, v in (("a", 1), ("b", 2)):
    p.writeString(k)
    p.writeI32(v)
p.writeMapEnd()
text = out.getvalue().decode()
assert json.loads(text) == ["str", "i32", 2, {"a": 1, "b": 2}], text
print("Example 4 OK:", text)

## Example 5: The error for a truncated message

In [ ]:
from thrift.protocol.TBinaryProtocol import TBinaryProtocolAcceleratedFactory
from thrift.transport.TTransport import TTransportException
from thrift.TSerialization import deserialize, serialize

from thrift.Thrift import TType
class Point:  # a hand-written struct in the shape `thrift --gen py` emits
    thrift_spec = (None, (1, TType.I32, "x", None, None), (2, TType.STRING, "name", "UTF8", None))
    def __init__(self, x=None, name=None): self.x, self.name = x, name
    def write(self, p): p.trans.write(p._fast_encode(self, [Point, self.thrift_spec]))
    def read(self, p): p._fast_decode(self, p, [Point, self.thrift_spec])

# Example 5: decoding bytes cut short raises a Thrift error, not a crash
factory = TBinaryProtocolAcceleratedFactory()
blob = serialize(Point(1, "a much longer name than fits"), protocol_factory=factory)
try:
    deserialize(Point(), blob[:9], protocol_factory=factory)
    raise AssertionError("expected an error for a truncated message")
except (TTransportException, EOFError) as error:
    print("Example 5 OK: expected error:", type(error).__name__)